## MPS Magnetization — 2D ITF Trotter Evolution

Time-evolve a product state under the 2D transverse-field Ising model using one clear MPS optimizer path.

This notebook keeps the geometry and mapping visible:

1. build the 2D square-lattice edges,
2. map those 2D edges to 1D MPS sites with one `OneDMap`,
3. build an explicit 2nd-order Trotter gate stream,
4. evolve with `MpsOptimizer(mode="dmrg")`.

**Parameters**:
- 4x4 square lattice, PBC
- H = J ΣZZ + h ΣX,  J = -1, h = 2
- 2nd-order Trotter with dt = 0.25
- Initial state: intermediate-temperature quench (arXiv:2503.20870)


In [ ]:
import math
import numpy as np
import quimb.tensor as qtn
import pepsy as py
from pepsy import tensors
from helper import (
    build_lattice, build_initial_state,
    build_mpo_z_sq, measure_energy, measure_z, measure_z_sq,
)

try:
    import torch
except Exception:
    torch = None


### Backend selection

Uncomment the backend you want to use.

In [ ]:
# ── Backend (pick one) ───────────────────────────────────────────────────────
backend = "cupy"
# backend = "torch"

if backend == "cupy":
    to_backend = tensors.backend_cupy(dtype="complex128")
elif backend == "torch":
    DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
    to_backend = tensors.backend_torch(device=DEVICE, dtype=torch.complex128)
else:
    raise ValueError(f"Unknown backend: {backend}")

optimizer = tensors.build_optimizer(progbar=False, directory="cash/", parallel=True)

### Lattice, Mapping, And Hamiltonian

`build_lattice(...)` uses Pepsy's ITF lattice builder. For `lattice="square"`, Pepsy delegates the 2D edge generation to `qtn.edges_2d_square(...)`, then maps every 2D coordinate to the 1D MPS chain with the single `mapper` below.


In [ ]:
# -- Physical parameters -----------------------------------------------------
Lx, Ly = 4, 4
L = Lx * Ly
coupling_j = -1.0   # J
field_h = +2.0      # h
cyclic = True
lattice = "square"
dt = 0.25           # Trotter step size
mapping_mode = "hilbert"

# -- Build lattice, MPO Hamiltonian, and one shared 2D <-> 1D mapper ----------
lat = build_lattice(
    Lx,
    Ly,
    coupling_j,
    field_h,
    cyclic=cyclic,
    lattice=lattice,
    mode=mapping_mode,
)
mpo_H = lat["mpo_H"]
edges_2d = lat["edges_2d"]
edges_1d = lat["edges_1d"]
sites = lat["sites"]
mapper = lat["mapper"]
one_d_to_two_d = lat["one_d_to_two_d"]
two_d_to_one_d = lat["two_d_to_one_d"]

# Same square-lattice edges, checked directly against quimb's generator.
if lattice == "square":
    edges_2d_direct = tuple(
        tuple(tuple(site) for site in edge)
        for edge in qtn.edges_2d_square(Lx, Ly, cyclic=cyclic)
    )
    assert {frozenset(edge) for edge in edges_2d} == {
        frozenset(edge) for edge in edges_2d_direct
    }

# Same mapping used for Hamiltonian MPO and for the Trotter gate locations.
edges_1d_from_map = tuple(
    (two_d_to_one_d[tuple(site_a)], two_d_to_one_d[tuple(site_b)])
    for site_a, site_b in edges_2d
)
assert edges_1d == edges_1d_from_map
assert sites == tuple((site,) for site in range(L))

mapper.show(title=f"{mapping_mode} 2D -> 1D mapping")
print(f"Lattice: {Lx}x{Ly}, L={L}, {'PBC' if cyclic else 'OBC'}")
print(f"H = J*ΣZZ + h*ΣX,  J={coupling_j}, h={field_h}")
print(f"Mapping mode: {mapping_mode}")
print(f"2D edges: {len(edges_2d)} | mapped 1D edges: {len(edges_1d)}")
print(f"First 2D edge {edges_2d[0]} -> 1D edge {edges_1d[0]}")
print(f"Trotter dt = {dt}")


### Initial state

Product state |Ψ(θ)⟩ from arXiv:2503.20870:
θ = arcsin(h/(zJ)) + 2π/9

In [ ]:
# ── Initial state ────────────────────────────────────────────────────────────
theta_offset = 2 * math.pi / 9   # offset from arcsin(h/(Jz)); change to set custom angle

psi0, theta_paper = build_initial_state(L, coupling_j, field_h, theta_offset=theta_offset)
print(
    f"Initial state: |Ψ(θ)⟩ product state, "
    f"θ={theta_paper:.4f} rad ({math.degrees(theta_paper):.2f}°), "
    f"offset={theta_offset:.4f}"
)

### Explicit 2nd-Order Trotter Gates

For one Trotter step we use Strang splitting:

1. apply the X half-step on every site,
2. apply the ZZ full-step on every mapped lattice edge,
3. apply the X half-step again.

Pepsy's `rx(theta)` convention is `exp(-i theta X / 2)`, so the half-step `exp(-i h dt X / 2)` is `py.rx(h * dt)`. Likewise, `py.rzz(theta)` is `exp(-i theta ZZ / 2)`, so the full bond step `exp(-i J dt ZZ)` is `py.rzz(2 * J * dt)`.


In [ ]:
# -- Build one explicit 2nd-order Trotter gate stream ------------------------
rx_half = to_backend(py.rx(field_h * dt))
rzz_full = to_backend(py.rzz(2.0 * coupling_j * dt))

gates_x_left = [(rx_half, site) for site in sites]
gates_zz = [(rzz_full, edge) for edge in edges_1d]
gates_x_right = [(rx_half, site) for site in sites]

gates_trotter = gates_x_left + gates_zz + gates_x_right

print(f"X half-step gates: {len(gates_x_left)}")
print(f"ZZ edge gates:     {len(gates_zz)}")
print(f"X half-step gates: {len(gates_x_right)}")
print(f"Total gates:       {len(gates_trotter)}")
print(f"Example one-site gate location: {gates_x_left[0][1]}")
print(f"Example two-site gate location: {gates_zz[0][1]}")


### MPS Evolution Parameters

Keep one optimizer mode in focus here: `dmrg`. The other MPS modes are useful diagnostics, but comparing modes is a separate experiment from checking the lattice mapping and gate stream.


In [ ]:
# -- Evolution parameters ----------------------------------------------------
n_steps = 20                    # number of Trotter steps
chi = 120                       # MPS bond dimension
mode = "dmrg"

# DMRG-specific controls:
n_iter = 5                      # inner FIT iterations per 2q gate
k_2q_batch = 1                  # batch this many 2q gates into one FIT window

print(f"n_steps = {n_steps}, T = {n_steps * dt}, chi = {chi}")
print(f"Mode: {mode}")


### Build Z² MPO

In [ ]:
# ── Build Z² MPO ─────────────────────────────────────────────────────────────
mpo_z_sq_offdiag, diagonal_shift, mpo_z = build_mpo_z_sq(Lx, Ly, mapper)
print(f"M = (1/L)ΣZ_i  →  MPO bond dim: {mpo_z.max_bond()}")
print(f"M² off-diag MPO bond dim: {mpo_z_sq_offdiag.max_bond()}")

### Run time evolution

In [ ]:
from tqdm import tqdm

mpo_H.apply_to_arrays(to_backend)
mpo_z.apply_to_arrays(to_backend)
mpo_z_sq_offdiag.apply_to_arrays(to_backend)

psi = psi0.copy()
psi.apply_to_arrays(to_backend)

engine = py.MpsOptimizer(psi, chi=chi, mode=mode, contraction_opt=optimizer)
engine.set_gates(gates_trotter)

times = [0.0]
energy = [measure_energy(engine.p.copy(), mpo_H, L, optimizer)]
z_sq = [measure_z_sq(engine.p.copy(), mpo_z_sq_offdiag, diagonal_shift, optimizer)]
z_mag = [measure_z(engine.p.copy(), mpo_z, optimizer)]
norm_proxy_per_step = []

pbar = tqdm(range(n_steps), desc=mode)
for step in pbar:
    engine.run(
        progbar=False,
        fidelity_samples=10,
        n_iter=n_iter,
        k_2q_batch=k_2q_batch,
    )

    t_now = (step + 1) * dt
    psi_t = engine.p.copy()
    times.append(t_now)
    energy.append(measure_energy(psi_t, mpo_H, L, optimizer))
    z_sq.append(measure_z_sq(psi_t, mpo_z_sq_offdiag, diagonal_shift, optimizer))
    z_mag.append(measure_z(psi_t, mpo_z, optimizer))

    loss = engine.losses[-1]
    norm_proxy_per_step.append(loss)
    pbar.set_postfix({
        "norm": f"{loss:.6f}",
        "E/L": f"{energy[-1]:.4f}",
        "Z^2": f"{z_sq[-1]:.4f}",
        "Z": f"{z_mag[-1]:.4f}",
    })

times = np.array(times)
energy = np.array(energy)
z_sq = np.array(z_sq)
z_mag = np.array(z_mag)
norm_proxy_per_step = np.array(norm_proxy_per_step)
print(f"\nDone. {len(times)} measurement points.")


### Plot results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "lines.linewidth": 1.8,
    "lines.markersize": 4,
})

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)

axes[0].plot(times, energy, "o-", color="#1f77b4", markersize=3)
axes[0].set_ylabel(r"$E / L$")
axes[0].set_title("Energy per site")

axes[1].plot(times, z_sq, "s-", color="#d62728", markersize=3)
axes[1].set_ylabel(r"$\langle (\Sigma Z / L)^2 \rangle$")
axes[1].set_title("Magnetization squared")

axes[2].plot(times, z_mag, "^-", color="#2ca02c", markersize=3)
axes[2].set_ylabel(r"$\langle \Sigma Z / L \rangle$")
axes[2].set_title("Magnetization")

for ax in axes:
    ax.set_xlabel(r"$t$")
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    f"MPS Trotter ({mode}) — {Lx}×{Ly} ITF, $dt$={dt}, $\\chi$={chi}, "
    f"{'PBC' if cyclic else 'OBC'}",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.savefig("magnetization.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ── Norm proxy per Trotter step ──────────────────────────────────────────────
step_indices = np.arange(1, n_steps + 1)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(step_indices, norm_proxy_per_step, ">-", lw=1.2, markersize=4,
        color="#7b2cbf", alpha=0.85)
ax.axhline(norm_proxy_per_step.mean(), ls="--", lw=0.8, color="gray", alpha=0.6,
           label=f"mean = {norm_proxy_per_step.mean():.6f}")

# Auto y-limits to show variations clearly
ymin = norm_proxy_per_step.min() - 0.005
ymax = min(norm_proxy_per_step.max() + 0.005, 1.0)
ax.set_ylim(ymin, ymax)

ax.set_xlabel("Trotter step")
ax.set_ylabel("Norm proxy")
ax.set_title(f"Norm proxy per step — {mode}, $\\chi$={chi}, $dt$={dt}",
             fontweight="bold")
ax.grid(True, alpha=0.25, linestyle="--")
ax.set_xlim(0.5, n_steps + 0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="lower left", framealpha=0.9)
plt.tight_layout()
plt.show()

print(f"Norm proxy per step: min={norm_proxy_per_step.min():.8f}, "
      f"max={norm_proxy_per_step.max():.8f}, mean={norm_proxy_per_step.mean():.8f}")

### Save results

In [ ]:
# import quimb as qu

# results = {
#     "times": times,
#     "energy": energy,
#     "z_sq": z_sq,
#     "z_mag": z_mag,
#     "losses": np.array(engine.losses),
#     "Lx": Lx, "Ly": Ly, "L": L,
#     "J": coupling_j, "h": field_h,
#     "dt": dt, "chi": chi, "mode": mode,
#     "cyclic": cyclic, "n_steps": n_steps,
# }

# save_path = f"store/mps_mag_L{L}_dt{dt}_c{cyclic}_chi{chi}_{mode}.pkl"
# qu.save_to_disk(results, save_path)
# print(f"Saved to {save_path}")

### Sample from final MPS

In [ ]:
from pepsy.sampling import MpsSampler

# Sample from the final MPS
sampler = MpsSampler(
    engine.p,
    one_d_to_two_d=lat["res"]["one_d_to_two_d"],
)
result = sampler.sample(n_samples=8, seed=2)

# Plot samples as 2D heatmaps
n_show = min(len(result), 8)
ncols = 8
nrows = (n_show + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3.2 * nrows))
axes = np.atleast_2d(axes)

cmap = plt.cm.colors.ListedColormap(["#2196F3", "#FF5722"])  # blue=↑, red=↓

for idx in range(nrows * ncols):
    ax = axes[idx // ncols, idx % ncols]
    if idx < n_show:
        grid = result.configs_2d[idx]
        prob = result.probs[idx]
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=1, aspect="equal")
        for iy in range(Ly):
            for ix in range(Lx):
                label = "↑" if grid[iy, ix] == 0 else "↓"
                ax.text(ix, iy, label, ha="center", va="center",
                        fontsize=14, fontweight="bold", color="white")
        ax.set_title(f"p = {prob:.3e}", fontsize=10)
        ax.set_xticks(range(Lx))
        ax.set_yticks(range(Ly))
        ax.set_xticklabels(range(Lx), fontsize=8)
        ax.set_yticklabels(range(Ly), fontsize=8)
        ax.set_xlabel("x", fontsize=9)
        ax.set_ylabel("y", fontsize=9)
    else:
        ax.axis("off")

fig.suptitle(
    f"Sampled spin configurations — {Lx}×{Ly} ITF, t={times[-1]:.2f}, χ={chi}",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Summary
probs = np.array(result.probs)
mags = result.magnetizations()
print(f"Sampled {len(result)} configs | prob range: [{probs.min():.3e}, {probs.max():.3e}]")
print(f"Sample magnetizations: {mags}")